# 03 — Modeling e Hyperparameter Tuning

## Obiettivi didattici

1. Confrontare tre famiglie di modelli (lineare, ensemble bagging, gradient boosting).
2. Applicare **K-fold cross-validation** con scoring corretto (RMSE).
3. Eseguire **Grid/Randomized search** su iperparametri rilevanti.
4. Avvolgere il target con `TransformedTargetRegressor(log1p, expm1)` per gestire automaticamente la trasformazione.


In [1]:
import sys; sys.path.insert(0, '../src')
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd

from ames_pipeline.data import load_raw
from ames_pipeline.wrangling import fill_structural_missing, remove_grliv_area_outliers
from ames_pipeline.features import AmesFeatureEngineer
from ames_pipeline.preprocessing import build_preprocessor, infer_column_groups
from ames_pipeline.models import get_all_pipelines
from ames_pipeline.tuning import tune_all_models, summarize_tuning, wrap_with_log_target
from ames_pipeline.config import DEFAULT_CONFIG, PipelineConfig
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score, KFold


## Setup: dati + pipeline candidate

Riproduciamo brevemente il flusso dei notebook precedenti.

In [2]:
df = load_raw()
df = fill_structural_missing(df)
df = remove_grliv_area_outliers(df)
X = df.drop(columns=['Order','PID','SalePrice'])
y = df['SalePrice']

# Stratificazione su quintili di prezzo per stabilità
y_bins = pd.qcut(y, q=5, labels=False, duplicates='drop')
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y_bins,
)
print(f'train={len(X_train)}, test={len(X_test)}')

fe = AmesFeatureEngineer()
X_train_fe = fe.fit_transform(X_train)
groups = infer_column_groups(X_train_fe)
preprocessor = build_preprocessor(
    numeric_cols=groups['numeric'],
    ordinal_cols=groups['ordinal'],
    nominal_cols=groups['nominal'],
)

# Antepone feature_engineer alle pipeline candidate
base = get_all_pipelines(preprocessor)
candidates = {
    name: Pipeline(steps=[('feature_engineer', AmesFeatureEngineer())] + list(p.steps))
    for name, p in base.items()
}
list(candidates.keys())


train=2341, test=586


['Ridge', 'RandomForest', 'XGBoost']

## Baseline cross-validation (no tuning)

Misura le performance dei modelli con iperparametri di default. Serve come riferimento per stimare il guadagno del tuning successivo.

In [3]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)
baseline = []
for name, pipe in candidates.items():
    wrapped = wrap_with_log_target(pipe)
    rmse = -cross_val_score(wrapped, X_train, y_train,
                            scoring='neg_root_mean_squared_error',
                            cv=cv, n_jobs=-1)
    baseline.append({'model': name, 'rmse_mean': rmse.mean(), 'rmse_std': rmse.std()})
pd.DataFrame(baseline).sort_values('rmse_mean')

,model,rmse_mean,rmse_std
0,Ridge,20997.695625,1797.600680
2,XGBoost,21294.564063,1271.861160
1,RandomForest,25196.150648,2300.125482


## Tuning iperparametri

**Strategia per modello**:

- **Ridge**: `GridSearchCV` su 6 valori di α (10⁻¹ → 10²) — grid piccola.
- **RandomForest**: `GridSearchCV` su 24 combinazioni — esauriente.
- **XGBoost**: `RandomizedSearchCV` 30 sample su grid combinatoria di   ~200 combinazioni — efficiente e nella pratica raggiunge il 95-99%   del best score di GridSearch completa.

**Scoring**: `neg_root_mean_squared_error` su `log1p(target)` (gestito automaticamente da `TransformedTargetRegressor`).

Tempo atteso ~5-10 minuti su laptop senza GPU. Per smoke-test ridurre le grid in `config.py`.

In [4]:
config = DEFAULT_CONFIG
results = tune_all_models(
    pipelines=candidates,
    X=X_train, y=y_train,
    config=config,
    xgb_n_iter=30,
)
summary = summarize_tuning(results)
summary

Fitting 5 folds for each of 6 candidates, totalling 30 fits


Fitting 5 folds for each of 24 candidates, totalling 120 fits


Fitting 5 folds for each of 30 candidates, totalling 150 fits


,model,rmse_log_cv,duration_s,best_params
0,Ridge,20373.608753,1.011845,{'model__alpha': 100.0}
1,XGBoost,20461.922266,144.260976,"{'model__subsample': 0.8, 'model__reg_lambda':..."
2,RandomForest,22961.475417,51.315536,"{'model__max_depth': 20, 'model__max_features'..."


## Discussione iperparametri ottimi

- **Ridge α**: `α=10` tipico. Valori troppo piccoli sovra-adattano ai rumori di OneHot; troppo grandi schiacciano il segnale.
- **RandomForest**: con `max_depth=None` (alberi profondi) e `max_features='sqrt'` ottiene il miglior bias-variance trade-off.
- **XGBoost**: `learning_rate=0.05` + `n_estimators=800` è il classico binomio (lr basso compensato da più alberi). `max_depth=5` evita l'overfit, `subsample=0.8` introduce regolarizzazione stocastica.


## Esportiamo le pipeline migliori per il notebook successivo

In [5]:
import joblib
from pathlib import Path
out_dir = Path('../reports/models')
out_dir.mkdir(parents=True, exist_ok=True)
for name, r in results.items():
    path = out_dir / f'{name.lower()}_best.joblib'
    joblib.dump(r.best_estimator, path)
    print(f'salvato: {path.relative_to(Path(".."))}  RMSE_log_cv={r.best_score:.4f}')


salvato: reports/models/ridge_best.joblib  RMSE_log_cv=20373.6088
salvato: reports/models/randomforest_best.joblib  RMSE_log_cv=22961.4754
salvato: reports/models/xgboost_best.joblib  RMSE_log_cv=20461.9223
